# Transformer — next-hour CO₂ and NOx forecasting (Plant Bowen)

This notebook is the interactive version of `train_transformer.py`. The data pipeline comes from `src/data_prep.py`, the figures from `src/plots.py`, and the network, training target and metrics from `train_transformer.py`. Nothing is redefined here, so the notebook and the script always train the same model on the same data.

**Kernel:** select the project's `.venv` interpreter. Training runs on CPU and takes several minutes per pollutant.

## 1. Setup

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# Works whether the notebook runs from the project root (VS Code default here)
# or from inside notebooks/.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
for p in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from IPython.display import Image, display

import data_prep
import plots
import train_transformer as tt

# Where outputs go. These are the same folders the script writes to, so
# re-running this notebook overwrites the script's Transformer results and figures.
RESULTS_DIR = PROJECT_ROOT / "results"
plots.FIG_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
plots.FIG_DIR.mkdir(exist_ok=True)

keras.utils.set_random_seed(tt.SEED)
print(f"TensorFlow {tf.__version__} | seed {tt.SEED} | "
      f"lookback {data_prep.LOOKBACK} h | horizon {data_prep.HORIZON} h")
print(f"Hyperparameters: embed={tt.EMBED_DIM}, heads={tt.NUM_HEADS}, "
      f"ff={tt.FEED_FORWARD_DIM}, blocks={tt.NUM_BLOCKS}, dropout={tt.DROPOUT}, "
      f"lr={tt.LEARNING_RATE}, batch={tt.BATCH_SIZE}, max epochs={tt.EPOCHS}")


def show(name: str, width: int = 760) -> None:
    # plots.py saves figures to disk instead of drawing them inline, so we
    # display the saved PNG.
    display(Image(filename=str(plots.FIG_DIR / f"{name}.png"), width=width))


def slug(target: str) -> str:
    return "co2" if "CO2" in target else "nox"

## 2. Data

These are the same windows, splits and scalers as the ANN notebook, built by `data_prep.prepare()`. The dataset figures are already produced by the ANN notebook and are identical here, so they are not repeated.

In [ ]:
data = {target: data_prep.prepare(target) for target in data_prep.TARGETS}

### The training target: change since the last hour

Hourly emissions are highly persistent, so "next hour = this hour" is a strong baseline. A model that predicts the absolute level (about 1,400 tons of CO₂) has to rebuild that number from scaled inputs every hour. In steady operation the original level-predicting Transformer missed by about 29 tons where persistence missed by about 3, and so it lost to persistence on MAE.

The model therefore predicts the change $\Delta_t = y_t - y_{t-1}$, standardised on the training set. The forecast is $\hat y_t = y_{t-1} + \hat\Delta_t$, so an output of zero reproduces persistence exactly, and the network only has to learn the departures from it. This uses no extra information, because $y_{t-1}$ is already the last row of every input window. The ANN uses exactly the same target (`data_prep.change_targets()`), so the comparison isolates architecture.

In [ ]:
for target in data_prep.TARGETS:
    dt = data[target]
    raw_change = dt["y_train"] - dt["persistence_train"]
    print(f"{target}: level std {dt['y_train'].std():,.1f} | "
          f"hour-to-hour change std {raw_change.std():,.1f} | "
          f"median |change| {np.median(np.abs(raw_change)):,.1f}")

## 3. Model

The model is a two-block pre-norm Transformer encoder over the 24-hour window, with learned positional embeddings. Two choices come from the validation experiments:

- **Pooling:** the regression head sees the last hour's encoding next to the average over the window. The last hour carries most of the signal for a one-hour-ahead forecast, and averaging alone dilutes it.
- **Loss:** Huber instead of MSE. Huber is squared for small errors and linear for large ones, so rare startup and shutdown ramps don't dominate training.

In [ ]:
d = data[data_prep.TARGETS[0]]
example = tt.build_transformer(d["lookback"], d["X_train"].shape[2])
example.summary()

## 4. Training

There is one model per pollutant. `tt.train()` sets the seed, builds the network and fits it on the scaled change. Early stopping watches validation loss and restores the best weights, and the learning rate halves when validation loss plateaus.

In [ ]:
models, histories, change_scalers, train_seconds = {}, {}, {}, {}

for target in data_prep.TARGETS:
    print(f"\n{'#' * 70}\n#  {tt.MODEL_NAME} — {target}\n{'#' * 70}")
    (models[target], histories[target],
     change_scalers[target], train_seconds[target]) = tt.train(data[target])
    print(f"Finished in {train_seconds[target]:.1f}s after "
          f"{len(histories[target].history['loss'])} epochs")

## 5. Evaluation on the test set

This is the first time the test set is used. Predictions are converted to real units (last observed value plus predicted change) before scoring against persistence.

In [ ]:
predictions, metrics = {}, {}
rows = []
for target in data_prep.TARGETS:
    dt = data[target]
    predictions[target] = tt.forecast(models[target], dt, change_scalers[target])
    metrics[target] = tt.evaluate(dt["y_test"], predictions[target],
                                  dt["persistence_test"])
    for key in ["mae", "rmse", "mape", "r2"]:
        m, b = metrics[target]["model"][key], metrics[target]["persistence"][key]
        rows.append({
            "target": target, "metric": key.upper(),
            tt.MODEL_NAME: m, "Persistence": b,
            "improvement vs persistence (%)":
                None if key == "r2" else (b - m) / b * 100,
        })

pd.DataFrame(rows).set_index(["target", "metric"]).style.format(
    {tt.MODEL_NAME: "{:,.3f}", "Persistence": "{:,.3f}",
     "improvement vs persistence (%)": "{:+.1f}"}, na_rep="—")

### Where the errors come from

The test hours are split at the median size of the hour-to-hour change. In **steady** hours little changes and persistence is almost perfect. In **ramping** hours load is moving, which is where a forecaster can add value.

In [ ]:
rows = []
for target in data_prep.TARGETS:
    dt = data[target]
    y, base, pred = dt["y_test"], dt["persistence_test"], predictions[target]
    steady = np.abs(y - base) < np.median(np.abs(y - base))
    for label, mask in [("steady half", steady), ("ramping half", ~steady)]:
        rows.append({"target": target, "hours": label,
                     f"{tt.MODEL_NAME} MAE": np.abs(y - pred)[mask].mean(),
                     "Persistence MAE": np.abs(y - base)[mask].mean()})
pd.DataFrame(rows).set_index(["target", "hours"]).style.format("{:,.1f}")

### Model figures

There are six figures per pollutant: the training curves (loss is on the scaled hour-to-hour change), two weeks of test-set predictions, predicted vs actual, residuals, the comparison with the baseline, and errors broken down by hour and month.

In [ ]:
for target in data_prep.TARGETS:
    dt, pred, m = data[target], predictions[target], metrics[target]
    plots.fig_training_history(histories[target], target, tt.MODEL_NAME)
    plots.fig_prediction_timeseries(dt["ts_test"], dt["y_test"], pred, target, tt.MODEL_NAME)
    plots.fig_scatter(dt["y_test"], pred, target, tt.MODEL_NAME, m["model"]["r2"])
    plots.fig_residuals(dt["y_test"], pred, target, tt.MODEL_NAME)
    plots.fig_baseline_comparison(m, target, tt.MODEL_NAME)
    plots.fig_error_breakdown(dt["ts_test"], dt["y_test"], pred, target, tt.MODEL_NAME)

In [ ]:
FIGURE_KINDS = [("07", "training_history"), ("08", "timeseries"), ("09", "scatter"),
                ("10", "residuals"), ("11", "vs_baseline"), ("12", "error_breakdown")]
for target in data_prep.TARGETS:
    print(f"\n=== {target} ===")
    for number, kind in FIGURE_KINDS:
        show(f"{number}_{slug(target)}_{tt.MODEL_NAME.lower()}_{kind}")

## 6. Save

The saved files have the same names and format as the script's output. Note that the saved `.keras` model outputs the scaled change, not the level. Use `tt.forecast()` with the change scaler (from `data_prep.change_targets()`) to get forecasts in real units.

In [ ]:
results = []
for target in data_prep.TARGETS:
    dt, s = data[target], slug(target)
    models[target].save(RESULTS_DIR / f"transformer_{s}.keras")
    np.savez_compressed(
        RESULTS_DIR / f"transformer_{s}_predictions.npz",
        ts=dt["ts_test"], y_true=dt["y_test"], y_pred=predictions[target],
        persistence=dt["persistence_test"],
    )
    results.append({
        "target": target,
        "model": tt.MODEL_NAME,
        "epochs_run": len(histories[target].history["loss"]),
        "train_seconds": round(train_seconds[target], 1),
        "parameters": int(models[target].count_params()),
        "n_train": int(len(dt["y_train"])),
        "n_val": int(len(dt["y_val"])),
        "n_test": int(len(dt["y_test"])),
        "metrics": metrics[target],
    })

(RESULTS_DIR / "transformer_metrics.json").write_text(json.dumps(results, indent=2))
print("Saved models, predictions and metrics to", RESULTS_DIR)